# 2025 Colorado State University Hackathon Starter Notebook

This notebook will offer an example for loading geospatial data into a potentially desirable format that can be used for completing the challenges.

_________________________________

In [1]:
!pip install rasterio

In [2]:
import rasterio
import numpy as np


In [3]:
# Paths to your raster files
hillshade_path = "data/South_Clear_Creek/Lidar_DEM_Hillshade/South_Clear_Creek_BareEarth_Hillshade_1m.tif"
naip_path = "data/South_Clear_Creek/NAIP/South_Clear_Creek_2023_NAIP_1m.tif"
dem_path = "data/South_Clear_Creek/Lidar_DEM_Hillshade/South_Clear_Creek_BareEarth_DEM_1m.tif"
road_mask_path = "data/South_Clear_Creek/Roads_Boundary/South_Clear_Creek_Roads_Mask.tif"

In [4]:
# Initialize stacked_rasters as None
stacked_rasters = None

# Step 1: Load hillshade and stack it
with rasterio.open(hillshade_path) as hillshade_dataset:
    hillshade = hillshade_dataset.read(1)  # Read first band (hillshade)

# Stack hillshade array along the last axis (initial stacking)
stacked_rasters = np.stack([hillshade], axis=-1)

# Free up memory for hillshade
del hillshade

# Step 2: Load DEM and add it to the stacked array
with rasterio.open(dem_path) as dem_dataset:
    dem = dem_dataset.read(1)  # Read DEM band (bare earth elevation)

# Concatenate DEM to stacked_rasters along the last axis
stacked_rasters = np.concatenate([stacked_rasters, dem[..., np.newaxis]], axis=-1)

# Free up memory for DEM
del dem

# Step 3: Load NAIP bands (R, G, B, LiDAR) and add to stacked_rasters
with rasterio.open(naip_path) as naip_dataset:
    naip = naip_dataset.read([1, 2, 3, 4])  # Read 4 bands (R, G, B, LiDAR)

# Concatenate NAIP bands to stacked_rasters along the last axis
stacked_rasters = np.concatenate([stacked_rasters, naip[0][..., np.newaxis], 
                                  naip[1][..., np.newaxis], 
                                  naip[2][..., np.newaxis], 
                                  naip[3][..., np.newaxis]], axis=-1)

# Free up memory for NAIP
del naip

# Print the final shape of the stacked raster
print(f"Final stacked shape: {stacked_rasters.shape}")


Final stacked shape: (13115, 10460, 6)


In [5]:
with rasterio.open(road_mask_path) as road_mask_dataset:
    road_mask = road_mask_dataset.read(1)  # Read road mask (binary mask)

print(f"road_mask shape: {road_mask.shape}")

road_mask shape: (13115, 10460)


In [6]:
for i in range(6):
    print(f"Band {i} of Stacked Rasters:\n", stacked_rasters[..., i])


Band 0 of Stacked Rasters:
 [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Band 1 of Stacked Rasters:
 [[3.39999995e+38 3.39999995e+38 3.39999995e+38 ... 3.39999995e+38
  3.39999995e+38 3.39999995e+38]
 [3.39999995e+38 3.39999995e+38 3.39999995e+38 ... 3.39999995e+38
  3.39999995e+38 3.39999995e+38]
 [3.39999995e+38 3.39999995e+38 3.39999995e+38 ... 3.39999995e+38
  3.39999995e+38 3.39999995e+38]
 ...
 [3.39999995e+38 3.39999995e+38 3.39999995e+38 ... 3.39999995e+38
  3.39999995e+38 3.39999995e+38]
 [3.39999995e+38 3.39999995e+38 3.39999995e+38 ... 3.39999995e+38
  3.39999995e+38 3.39999995e+38]
 [3.39999995e+38 3.39999995e+38 3.39999995e+38 ... 3.39999995e+38
  3.39999995e+38 3.39999995e+38]]
Band 2 of Stacked Rasters:
 [[1.79e+308 1.79e+308 1.79e+308 ... 1.79e+308 1.79e+308 1.79e+308]
 [1.79e+308 1.79e+308 1.79e+308 ... 1.79e+308 1.79e+308 1.79e+308]
 [1.79e+308 1.79e+308 1.79

In [7]:
mid_row = stacked_rasters.shape[0] // 2
mid_col = stacked_rasters.shape[1] // 2

print(mid_row)
print(mid_col)

mid_values = stacked_rasters[mid_row, mid_col, :]
print(f"Middle Stacked Raster Values at ({mid_row}, {mid_col}):", mid_values)


6557
5230
Middle Stacked Raster Values at (6557, 5230): [ 227.         3611.18164062  158.55555556  153.31944445  106.65277778
  168.86111111]


In [8]:
nonsense = stacked_rasters[0, 0, :]
print(f"nonsense Raster Values at ({0}, {0}):", nonsense)

nonsense Raster Values at (0, 0): [0.00000000e+000 3.39999995e+038 1.79000000e+308 1.79000000e+308
 1.79000000e+308 1.79000000e+308]


In [9]:
print("Stacked Rasters Shape:", stacked_rasters.shape)

Stacked Rasters Shape: (13115, 10460, 6)


In [11]:
import numpy as np

# Assuming stacked_rasters is already loaded and contains the data

# Step 1: Fix inf values by replacing them with NaN
stacked_rasters[np.isinf(stacked_rasters)] = np.nan  

# Step 2: Create a mask to identify valid values (ignore edge values greater than 3.4e+38)
valid_mask = ((stacked_rasters < 2.1e+37) & (stacked_rasters > 0))  # Ignore values that are considered invalid or edge values

# Step 3: Compute min/max only on valid values (using the valid_mask)
valid_min = np.nanmin(stacked_rasters[valid_mask])  # Min value from valid data
valid_max = np.nanmax(stacked_rasters[valid_mask])  # Max value from valid data

print("min:", valid_min)
print("max:", valid_max)

# Step 4: Replace NaNs (which were previously Inf values) with the valid min value
stacked_rasters[np.isnan(stacked_rasters)] = valid_min  

# Step 5: Clip values to the valid range (prevent overflow)
stacked_rasters = np.clip(stacked_rasters, valid_min, valid_max)

# Step 6: Normalize valid values only (scaled to [-1, 1])
stacked_rasters[valid_mask] = 2 * (stacked_rasters[valid_mask] - valid_min) / (valid_max - valid_min + 1e-8) - 1

# Step 7: Handle edge values (those that are not valid)
# Optionally replace edge values with 0 or NaN
stacked_rasters[~valid_mask] = 0  # Replace invalid values with 0 (or use np.nan if preferred)

# Optional: Print a portion of the data to check the result
print("Processed Stacked Rasters:")
print(stacked_rasters)


min: 4.0
max: 4210.41943359375
Processed Stacked Rasters:
[[[0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  ...
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  ...
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  ...
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]]

 ...

 [[0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  ...
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  ...
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  ...
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0.]]]


In [12]:
# Print middle region again to verify
mid_values = stacked_rasters[6557, 5230, :]
print(f"Middle Stacked Raster Values at (6557, 5230):", mid_values)

Middle Stacked Raster Values at (6557, 5230): [-0.89397158  0.71508415 -0.92651443 -0.92900402 -0.95119232 -0.92161452]


In [13]:
for i in range(6):
    print(f"Band {i} of Stacked Rasters:\n", stacked_rasters[..., i])


Band 0 of Stacked Rasters:
 [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Band 1 of Stacked Rasters:
 [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Band 2 of Stacked Rasters:
 [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Band 3 of Stacked Rasters:
 [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Band 4 of Stacked Rasters:
 [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Band 5 of Stacked Rasters:
 [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [

In [ ]:
# stacked_rasters = 2 * (stacked_rasters - valid_min) / (valid_max - valid_min + 1e-8) - 1
# # 

In [23]:
stacked_rasters.dtype

dtype('float64')

In [24]:
road_mask.dtype

dtype('float32')

In [25]:
np.save('stacked_rasters.npy', stacked_rasters)
np.save('road_mask.npy', road_mask)

In [14]:
import torch
from torch.utils.data import Dataset

class RoadDataset(Dataset):
    def __init__(self, stacked_rasters, road_masks):
        self.stacked_rasters = stacked_rasters  # Input features
        self.road_masks = road_masks  # Output/target masks

    def __len__(self):
        return len(self.stacked_rasters)  # Number of samples

    def __getitem__(self, idx):
        # Get the input and target for the specific index
        x = torch.tensor(self.stacked_rasters[idx], dtype=torch.float32)
        y = torch.tensor(self.road_masks[idx], dtype=torch.float32)
        return x, y

# Example usage
# Assuming `stacked_rasters` and `road_masks` are your input and target numpy arrays
train_dataset = RoadDataset(stacked_rasters, road_mask)

print("Training dataset created")


Training dataset created


In [15]:
# Print the first sample from the train_dataset
sample_input, sample_label = train_dataset[0]
print(f"Sample Input Shape: {sample_input.shape}")
print(f"Sample Label Shape: {sample_label.shape}")

# Optionally, print out the data values (or a portion of them) if you want to inspect them
print("Sample Input Data:", sample_input)
print("Sample Label Data:", sample_label)


Sample Input Shape: torch.Size([10460, 6])
Sample Label Shape: torch.Size([10460])
Sample Input Data: tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        ...,
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])
Sample Label Data: tensor([0., 0., 0.,  ..., 0., 0., 0.])


In [20]:
# Observation index
obs_index = 5230

# Get the sample from the dataset
sample_input, _ = train_dataset[6557]  # Ignore label for now

# Extract the 5230th observation and its 6 bands (all features for this observation)
observation_5230 = sample_input[obs_index, :]

print(f"Observation {obs_index} values (6 bands): {observation_5230}")


Observation 5230 values (6 bands): tensor([-0.8940,  0.7151, -0.9265, -0.9290, -0.9512, -0.9216])


In [ ]:
# Middle Stacked Raster Values at (6557, 5230): [-0.89397158  0.71508415 -0.92651443 -0.92900402 -0.95119232 -0.92161452]

In [21]:
import pickle


In [22]:
# Save the train_dataset to a file
with open('correct_train_dataset.pkl', 'wb') as f:
    pickle.dump(train_dataset, f)